In [13]:
from gurobipy import GRB, Model, quicksum
import gurobipy as gb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus
import random
import math
import scipy.stats as sp
import gurobipy as gp
from itertools import permutations
import ast

In [14]:
df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/delivery.csv')


In [15]:
delivery

,Unnamed: 0,Size,Depot,Customer_1,Customer_2,Customer_3,Customer_4,Customer_5,Customer_6,Customer_7,Customer_8,Customer_9,Customer_10,Customer_11,Customer_12,Customer_13,Customer_14,Customer_15
0,Customer_1,3614.258155,22.793623,0.000000,18.182319,6.987729,7.678954,13.694510,8.329835,12.811331,11.318991,15.708546,12.106011,14.568374,20.642632,15.469589,13.696646,20.036904
1,Customer_2,2084.033455,20.549610,18.182319,0.000000,13.441909,23.413009,12.379373,8.802658,14.771390,19.420864,10.554106,21.996704,9.272869,14.903864,20.722340,21.779559,18.747650
2,Customer_3,1675.128584,2.150799,6.987729,13.441909,0.000000,9.027693,15.271752,11.090113,17.949967,9.428343,16.277598,20.421898,15.631096,10.987299,7.610090,23.582960,18.073724
3,Customer_4,2686.814929,15.998685,7.678954,23.413009,9.027693,0.000000,13.529232,15.887557,16.872246,18.641042,15.578955,18.062265,15.809172,12.307230,15.296024,10.108355,13.404692
4,Customer_5,3842.413770,9.327903,13.694510,12.379373,15.271752,13.529232,0.000000,10.943846,17.210991,12.290085,10.398820,12.638817,19.039575,8.938190,19.624197,11.997656,16.242980
5,Customer_6,1543.327493,14.338588,8.329835,8.802658,11.090113,15.887557,10.943846,0.000000,11.083877,12.895180,21.809757,18.748572,11.346177,20.955126,12.382848,14.034171,10.275895
6,Customer_7,423.104792,4.716093,12.811331,14.771390,17.949967,16.872246,17.210991,11.083877,0.000000,13.921626,13.094882,10.247036,17.225741,13.112054,17.492188,14.286843,17.553653
7,Customer_8,772.089875,23.942911,11.318991,19.420864,9.428343,18.641042,12.290085,12.895180,13.921626,0.000000,11.909236,9.302199,18.923284,16.165367,11.989881,17.207899,20.410760
8,Customer_9,1159.807009,22.496673,15.708546,10.554106,16.277598,15.578955,10.398820,21.809757,13.094882,11.909236,0.000000,12.894249,16.853413,17.332640,22.532072,15.230985,15.076411
9,Customer_10,2286.050615,2.921730,12.106011,21.996704,20.421898,18.062265,12.638817,18.748572,10.247036,9.302199,12.894249,0.000000,15.286151,15.799132,12.360215,17.759712,16.609838


(a) The traditional vehicle routing problem (VRP) aims to minimize the number of vans used. How does this variant of the VRP differ, and what impact does it have on the model’s formulation?
•  The difference in the objective function: While traditional VRP might aim to minimize the number of vehicles or total distance, this variant focuses on balancing the load among all vans.
•  The impact on the model's formulation: You should detail how constraints (such as maximum deliveries per van, specific customer grouping rules, battery range constraints, etc.) influence the model, and discuss any additional decision variables that might be needed to ensure equitable load distribution.


(b) Why is this model considered a MILP (Mixed-Integer Linear Program)?
A MILP (Mixed-Integer Linear Program) is characterized by a formulation where both continuous and discrete (integer or binary) decision variables appear in a model whose objective function and constraints are expressed as linear functions. In this delivery routing problem variant, the formulation qualifies as a MILP for several reasons:
•	Mixed Decision Variables:
o	Binary/Integer Variables: The model includes decision variables that specify discrete choices, such as whether a particular customer is assigned to a given van, or whether a specific route is taken. For example, the constraints that enforce no more than two of customers 7–9 on the same van, require binary variables to capture the inclusion or exclusion of each customer.
o	Continuous Variables: It also involves continuous variables representing quantities like the total weight (load) carried by each van or the distance covered. These variables are subject to capacity and range constraints.
•	Linear Objective Function:
The primary goal is to minimize the differences in the load among the vans. This type of objective can be modeled with linear expressions, such as minimizing the maximum difference or balancing terms that are summed over vans.
•	Linear Constraints:
Every operational constraint in the model is expressed in linear form. These include:
o	Capacity Constraint: Ensuring that no van carries more than 15,000 lbs of packages.
o	Battery Range Constraint: Making sure that the total distance traveled does not exceed 254 km.
o	Customer-Specific Constraints: Such as forcing customers 10–12 to be assigned together, or linking the assignment of customer 1 to the assignment of either customer 13 or 14.
o	Maximum Deliveries Constraint: No van can make more than 5 deliveries; this can be captured with a simple linear inequality summing binary decision variables.


Question c answer in the word doc. 

d: (Minimum Decision Variables & Gurobi Method:
•	Counting Variables:
o	Use 3×m3×m binary variables xikxik for mm customers and 3 vans.
o	Add continuous variables for each van’s load (e.g., LkLk) and for the auxiliary variables (e.g., Lmax⁡Lmax, Lmin⁡Lmin) that help formulate the load difference.
•	Gurobi: The attribute model.NumVars returns the total number of decision variables once your MILP is built.

question e answer in the doc 

f

In [16]:


# Extract the data
customers = list(range(1, 16))  # Customers 1-15
vans = list(range(1, 4))  # 3 vans
weights = df['Size'].values
depot_distances = df['Depot'].values

# Create distance matrix
distances = {}
for i in range(15):
    for j in range(15):
        if i != j:
            distances[(i+1, j+1)] = df.iloc[i, j+3]

# Create the optimization model
model = gp.Model("FedEx_Routing")

# Decision Variables
x = model.addVars([(i, j, k) for i in customers for j in customers for k in vans if i != j], vtype=GRB.BINARY, name="route")
y = model.addVars([(i, k) for i in customers for k in vans], vtype=GRB.BINARY, name="assignment")
z = model.addVars(vans, lb=0, name="weight")
max_weight = model.addVar(lb=0, name="max_weight")
min_weight = model.addVar(lb=0, name="min_weight")

# Objective Function: Minimize the maximum difference in weights between vans
model.setObjective(max_weight - min_weight, GRB.MINIMIZE)

# Constraints
# 1. Each customer must be assigned to exactly one van
for i in customers:
    model.addConstr(gp.quicksum(y[i,k] for k in vans) == 1)

# 2. All vans must be used
for k in vans:
    model.addConstr(gp.quicksum(y[i,k] for i in customers) >= 1)

# 3. Weight constraints for each van
for k in vans:
    model.addConstr(z[k] == gp.quicksum(weights[i-1] * y[i,k] for i in customers))
    model.addConstr(z[k] <= 15000)  # Maximum capacity
    model.addConstr(max_weight >= z[k])
    model.addConstr(min_weight <= z[k])

# 4. Distance constraints for each van
for k in vans:
    model.addConstr(
        gp.quicksum(depot_distances[i-1] * y[i,k] for i in customers) +
        gp.quicksum(distances[(i,j)] * x[i,j,k] for i in customers for j in customers if i != j) <= 254
    )

# 5. Flow conservation constraints
for k in vans:
    for i in customers:
        model.addConstr(gp.quicksum(x[i,j,k] for j in customers if i != j) == y[i,k])
        model.addConstr(gp.quicksum(x[j,i,k] for j in customers if i != j) == y[i,k])

# 6. Special constraints
# No more than 2 customers from {7,8,9} on same van
for k in vans:
    model.addConstr(gp.quicksum(y[i,k] for i in [7,8,9]) <= 2)

# Customers 10,11,12 must be on same van
for k in vans:
    model.addConstr(y[10,k] == y[11,k])
    model.addConstr(y[11,k] == y[12,k])

# If customer 1 is assigned, at least one of 13 or 14 must be on that van
for k in vans:
    model.addConstr(y[1,k] <= y[13,k] + y[14,k])

# Customer 2 cannot be with 3,4,5
for k in vans:
    model.addConstr(y[2,k] + y[3,k] <= 1)
    model.addConstr(y[2,k] + y[4,k] <= 1)
    model.addConstr(y[2,k] + y[5,k] <= 1)

# Maximum 5 deliveries per van
for k in vans:
    model.addConstr(gp.quicksum(y[i,k] for i in customers) <= 5)

# Solve the model
model.optimize()

# Print detailed results
print("\n=== Optimization Results ===")
print(f"Status: {model.status}")
print(f"Objective Value (Maximum Weight Difference): {model.objVal:.2f} lbs")

print("\n=== Van Assignments and Weights ===")
for k in vans:
    assigned_customers = [i for i in customers if y[i,k].x > 0.5]
    total_weight = sum(weights[i-1] for i in assigned_customers)
    print(f"\nVan {k}:")
    print(f"Customers: {assigned_customers}")
    print(f"Total Weight: {total_weight:.2f} lbs")
    print("Individual Customer Weights:")
    for i in assigned_customers:
        print(f"  Customer {i}: {weights[i-1]:.2f} lbs")

print("\n=== Weight Distribution ===")
print(f"Maximum Van Weight: {max_weight.x:.2f} lbs")
print(f"Minimum Van Weight: {min_weight.x:.2f} lbs")
print(f"Weight Difference: {model.objVal:.2f} lbs")

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 147 rows, 680 columns and 2271 nonzeros
Model fingerprint: 0x8e1f3a81
Variable types: 5 continuous, 675 integer (675 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+04]
Presolve removed 12 rows and 24 columns
Presolve time: 0.02s
Presolved: 135 rows, 656 columns, 2100 nonzeros
Variable types: 5 continuous, 651 integer (651 binary)
Found heuristic solution: objective 5529.4740659
Found heuristic solution: objective 3900.8989203
Found heuristic solution: objective 2968.3912297

Root relaxation: objective 0.000000e+00, 129 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | I

g

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 147 rows, 680 columns and 2271 nonzeros
Model fingerprint: 0x8e1f3a81
Variable types: 5 continuous, 675 integer (675 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+04]
Presolve removed 12 rows and 24 columns
Presolve time: 0.02s
Presolved: 135 rows, 656 columns, 2100 nonzeros
Variable types: 5 continuous, 651 integer (651 binary)
Found heuristic solution: objective 5529.4740659
Found heuristic solution: objective 3900.8989203
Found heuristic solution: objective 2968.3912297

Root relaxation: objective 0.000000e+00, 129 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | I

In [20]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import networkx as nx

# -----------------------
# Data Extraction and Preprocessing
# -----------------------
df = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/delivery.csv')

# Define customers 1–15 and vans 1–3
customers = list(range(1, 16))
vans = list(range(1, 4))
weights = df['Size'].values
depot_distances = df['Depot'].values

# Create a dictionary for customer-to-customer distances.
# Assumes that columns 4 onward (0-indexed: column index 3+) of the CSV hold inter-customer distances.
distances = {}
for i in range(15):
    for j in range(15):
        if i != j:
            distances[(i+1, j+1)] = df.iloc[i, j+3]

# -----------------------
# Subtour Detection Function
# -----------------------
def find_subtours(edges):
    """
    Given a list of edges (tuples (i,j)), this function returns a list of subtours.
    A subtour is defined as a connected component of the route graph with fewer nodes than the set of
    customers assigned to that vehicle.
    """
    G = nx.Graph()
    G.add_edges_from(edges)
    subtours = []
    # Each connected component is a potential tour.
    for comp in nx.connected_components(G):
        # In our formulation, a full tour should visit all customers assigned to the van.
        # If the component is smaller, then it is a subtour.
        if len(comp) < len(G.nodes()):
            subtours.append(list(comp))
    return subtours

# -----------------------
# Callback for Lazy Subtour Elimination
# -----------------------
# Global counter to track how many lazy constraints are added.
lazy_counter = {"count": 0}

def mycallback(model, where):
    if where == GRB.Callback.MIPSOL:
        # For each van, examine the current solution.
        for k in vans:
            # Collect edges for vehicle k with x > 0.5 in the current solution.
            selected_edges = []
            for i in customers:
                for j in customers:
                    if i != j:
                        sol_val = model.cbGetSolution(x[i, j, k])
                        if sol_val > 0.5:
                            selected_edges.append((i, j))
            subtours = find_subtours(selected_edges)
            for subtour in subtours:
                # Add a lazy constraint: For subtour S, the sum of arcs among nodes in S must be <= |S|-1.
                model.cbLazy(gp.quicksum(x[i, j, k] for i in subtour for j in subtour if i != j) <= len(subtour) - 1)
                lazy_counter["count"] += 1
                print(f"Lazy constraint added for van {k} subtour: {subtour}")

# -----------------------
# Build the MILP Model
# -----------------------
model = gp.Model("FedEx_Routing_With_Callback")

# Decision Variables
# x[i,j,k] = 1 if van k travels directly from customer i to customer j.
x = model.addVars([(i, j, k) for i in customers for j in customers for k in vans if i != j],
                  vtype=GRB.BINARY, name="route")
# y[i,k] = 1 if customer i is assigned to van k.
y = model.addVars([(i, k) for i in customers for k in vans],
                  vtype=GRB.BINARY, name="assignment")
# z[k] = total weight delivered by van k.
z = model.addVars(vans, lb=0, name="weight")

# Auxiliary variables to capture maximum and minimum load across vans.
max_weight = model.addVar(lb=0, name="max_weight")
min_weight = model.addVar(lb=0, name="min_weight")

# -----------------------
# Objective: Minimize difference in loads
# -----------------------
model.setObjective(max_weight - min_weight, GRB.MINIMIZE)

# -----------------------
# Constraints
# -----------------------

# (1) Each customer is assigned to exactly one van.
for i in customers:
    model.addConstr(gp.quicksum(y[i, k] for k in vans) == 1, name=f"assign_{i}")

# (2) All vans must serve at least one customer.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in customers) >= 1, name=f"van_usage_{k}")

# (3) Weight (load) constraints for each van.
for k in vans:
    model.addConstr(z[k] == gp.quicksum(weights[i-1] * y[i, k] for i in customers), name=f"weight_calc_{k}")
    model.addConstr(z[k] <= 15000, name=f"capacity_{k}")
    model.addConstr(max_weight >= z[k], name=f"max_bound_{k}")
    model.addConstr(min_weight <= z[k], name=f"min_bound_{k}")

# (4) Distance constraints for each van:
for k in vans:
    model.addConstr(
        gp.quicksum(depot_distances[i-1] * y[i, k] for i in customers) +
        gp.quicksum(distances[(i, j)] * x[i, j, k] for i in customers for j in customers if i != j)
        <= 254,
        name=f"range_limit_{k}"
    )

# (5) Flow conservation: if customer i is served by van k, then the number of departures equals the assignment.
for k in vans:
    for i in customers:
        model.addConstr(gp.quicksum(x[i, j, k] for j in customers if i != j) == y[i, k],
                        name=f"flow_out_{i}_{k}")
        model.addConstr(gp.quicksum(x[j, i, k] for j in customers if i != j) == y[i, k],
                        name=f"flow_in_{i}_{k}")

# (6) Special operational constraints:
# - No more than 2 of customers 7, 8, 9 on the same van.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in [7, 8, 9]) <= 2, name=f"special_7_9_{k}")

# - Customers 10, 11, 12 must be assigned to the same van.
for k in vans:
    model.addConstr(y[10, k] == y[11, k], name=f"same_van_10_11_{k}")
    model.addConstr(y[11, k] == y[12, k], name=f"same_van_11_12_{k}")

# - If customer 1 is assigned, at least one of 13 or 14 must be on that van.
for k in vans:
    model.addConstr(y[1, k] <= y[13, k] + y[14, k], name=f"cust1_req_{k}")

# - Customer 2 cannot be with customers 3, 4, 5.
for k in vans:
    model.addConstr(y[2, k] + y[3, k] <= 1, name=f"cust2_3_{k}")
    model.addConstr(y[2, k] + y[4, k] <= 1, name=f"cust2_4_{k}")
    model.addConstr(y[2, k] + y[5, k] <= 1, name=f"cust2_5_{k}")

# - Maximum 5 deliveries per van.
for k in vans:
    model.addConstr(gp.quicksum(y[i, k] for i in customers) <= 5, name=f"max_deliv_{k}")

# -----------------------
# Set up for Lazy Constraints
# -----------------------
model.Params.LazyConstraints = 1

# -----------------------
# Optimize using the Callback
# -----------------------
model.optimize(mycallback)

# -----------------------
# Report Results
# -----------------------
print("\n=== Optimization Results (with Callback Lazy Constraints) ===")
print(f"Status: {model.status}")
print(f"Optimal Objective Value (Max Weight Difference): {model.objVal:.2f} lbs")
print(f"Total Number of Lazy Constraints Added: {lazy_counter['count']}")


Set parameter LazyConstraints to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D81)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
LazyConstraints  1

Optimize a model with 147 rows, 680 columns and 2271 nonzeros
Model fingerprint: 0x8e1f3a81
Variable types: 5 continuous, 675 integer (675 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+04]
Presolve removed 11 rows and 24 columns
Presolve time: 0.02s
Presolved: 136 rows, 656 columns, 2100 nonzeros
Variable types: 5 continuous, 651 integer (651 binary)
Lazy constraint added for van 1 subtour: [9, 3]
Lazy constraint added for van 1 subtour: [4, 6, 7]
Lazy constraint added for van 2 subtour: [12, 5]
Lazy constraint added for van 2 subtour: [10, 11, 13]
Lazy constraint added for van 3 subtour: [8, 1]
Lazy constra